In [1]:
import math
import torch
import torch.nn as nn
from google.colab import files

# ============================================================
# ATTENTION 1 : DENSE CAUSAL
# ============================================================

def Attention_1(Q, K, V, return_scores=True):
    single = Q.dim() == 2
    if single:
        Q, K, V = Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)

    d_k = Q.size(-1)
    T = Q.size(1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    mask = torch.triu(torch.ones(T, T, device=Q.device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask.unsqueeze(0), float("-inf"))

    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)

    if single:
        output = output.squeeze(0)
        scores = scores.squeeze(0)

    return output, scores if return_scores else None


# ============================================================
# ATTENTION 2 : SLIDING WINDOW
# ============================================================

def Attention_2(Q, K, V, Window_size, return_scores=True):
    single = Q.dim() == 2
    if single:
        Q, K, V = Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)

    B, T, d_k = Q.shape

    offsets = torch.arange(Window_size - 1, -1, -1, device=Q.device)
    positions = torch.arange(T, device=Q.device)
    key_indices = positions[:, None] - offsets[None, :]
    valid = (key_indices >= 0) & (key_indices < T)
    key_indices = key_indices.clamp(0, T - 1)

    K_window = K[:, key_indices, :]
    V_window = V[:, key_indices, :]

    scores = torch.einsum("btd,btwd->btw", Q, K_window) / math.sqrt(d_k)
    scores = scores.masked_fill(~valid.unsqueeze(0), float("-inf"))

    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.einsum("btw,btwd->btd", attn_weights, V_window)

    if return_scores:
        full_scores = torch.full((B, T, T), float("-inf"), device=Q.device, dtype=Q.dtype)
        full_scores.scatter_(2, key_indices.unsqueeze(0).expand(B, -1, -1), scores)
    else:
        full_scores = None

    if single:
        output = output.squeeze(0)
        if full_scores is not None:
            full_scores = full_scores.squeeze(0)

    return output, full_scores


# ============================================================
# ATTENTION 3 : BIGBIRD-LIKE
# ============================================================

def Attention_3(Q, K, V, Window_size, Global_size, Random_size, return_scores=True):
    single = Q.dim() == 2
    if single:
        Q, K, V = Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)

    B, T, d_k = Q.shape

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    pos = torch.arange(T, device=Q.device)
    rows = pos[:, None]
    cols = pos[None, :]

    local_mask = (cols <= rows) & (cols >= rows - Window_size + 1)

    random_indices = torch.randint(0, T, (T, Random_size), device=Q.device)
    random_mask = torch.zeros(T, T, device=Q.device, dtype=torch.bool)
    random_mask.scatter_(1, random_indices, True)

    preliminary_mask = local_mask | random_mask
    preliminary_scores = scores.masked_fill(~preliminary_mask.unsqueeze(0), 0.0)

    Global_scores = preliminary_scores.sum(dim=0).mean(dim=0)
    Global_indexes = torch.topk(Global_scores, Global_size).indices

    global_mask = torch.zeros(T, T, device=Q.device, dtype=torch.bool)
    global_mask[:, Global_indexes] = True
    global_mask = global_mask & (cols <= rows)

    allowed_mask = local_mask | random_mask | global_mask
    scores = scores.masked_fill(~allowed_mask.unsqueeze(0), float("-inf"))

    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)

    if single:
        output = output.squeeze(0)
        scores = scores.squeeze(0)

    return output, scores if return_scores else None


# ============================================================
# LOAD DATA
# ============================================================

uploaded = files.upload()

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Dataset length:", len(data))


# ============================================================
# MODEL PARAMETERS
# ============================================================

n_embd = 64
train_length = 128
batch_size = 32
n_pos_embd = 64

n_QK = 32
n_V = 64

Window_size = 16
Global_size = 5
Random_size = 5


# ============================================================
# EMBEDDINGS
# ============================================================

token_embedding = nn.Embedding(vocab_size, n_embd)
pos_embedding = nn.Embedding(train_length, n_pos_embd)


# ============================================================
# LAYER 1
# ============================================================

Qc = nn.Linear(n_embd, n_QK)
Kc = nn.Linear(n_embd, n_QK)
Vc = nn.Linear(n_embd, n_V)

FFN_1 = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd)
)


# ============================================================
# LAYER 2
# ============================================================

n_QK = 128
Qc_2 = nn.Linear(n_embd, n_QK)
Kc_2 = nn.Linear(n_embd, n_QK)
Vc_2 = nn.Linear(n_embd, n_V)

FFN_2 = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd)
)


# ============================================================
# LANGUAGE MODEL HEAD
# ============================================================

lm_head = nn.Linear(n_embd, vocab_size)


# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# MOVE TO GPU
# ============================================================

token_embedding = token_embedding.to(device)
pos_embedding = pos_embedding.to(device)

Qc = Qc.to(device)
Kc = Kc.to(device)
Vc = Vc.to(device)

Qc_2 = Qc_2.to(device)
Kc_2 = Kc_2.to(device)
Vc_2 = Vc_2.to(device)

FFN_1 = FFN_1.to(device)
FFN_2 = FFN_2.to(device)

lm_head = lm_head.to(device)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    list(token_embedding.parameters()) +
    list(pos_embedding.parameters()) +
    list(Qc.parameters()) +
    list(Kc.parameters()) +
    list(Vc.parameters()) +
    list(Qc_2.parameters()) +
    list(Kc_2.parameters()) +
    list(Vc_2.parameters()) +
    list(FFN_1.parameters()) +
    list(FFN_2.parameters()) +
    list(lm_head.parameters()),
    lr=1e-3
)


# ============================================================
# DEVICE CHECK
# ============================================================

print("Token embedding:", next(token_embedding.parameters()).device)
print("Qc:", next(Qc.parameters()).device)
print("Qc_2:", next(Qc_2.parameters()).device)
print("FFN_1:", next(FFN_1.parameters()).device)
print("FFN_2:", next(FFN_2.parameters()).device)
print("LM head:", next(lm_head.parameters()).device)


# ============================================================
# TRAINING
# ============================================================

num_steps = 20000

for step in range(num_steps):

    # RANDOM BATCH
    ix = torch.randint(0, len(data) - train_length - 1, (batch_size,))

    batch_input = torch.stack([
        data[i:i + train_length] for i in ix
    ]).to(device)

    targets = torch.stack([
        data[i + 1:i + train_length + 1] for i in ix
    ]).to(device)


    # ========================================================
    # EMBEDDING
    # ========================================================

    x = token_embedding(batch_input)

    pos_emb = pos_embedding(
        torch.arange(train_length, device=device)
    )

    x = x + pos_emb


    # ========================================================
    # LAYER 1
    # ========================================================

    Q_t = Qc(x)
    K_t = Kc(x)
    V_t = Vc(x)

    output_L1, _ = Attention_1(
        Q_t, K_t, V_t,
        return_scores=False
    )

    x_L1 = x + output_L1

    ffn_output_L1 = FFN_1(x_L1)

    x_L1_final = x_L1 + ffn_output_L1


    # ========================================================
    # LAYER 2
    # ========================================================

    Q_t2 = Qc_2(x_L1_final)
    K_t2 = Kc_2(x_L1_final)
    V_t2 = Vc_2(x_L1_final)

    output_L2, _ = Attention_1(
        Q_t2, K_t2, V_t2,
        return_scores=False
    )

    x_L2 = x_L1_final + output_L2

    ffn_output_L2 = FFN_2(x_L2)

    x_final = x_L2 + ffn_output_L2


    # ========================================================
    # LOGITS
    # ========================================================

    logits = lm_head(x_final)


    # ========================================================
    # LOSS
    # ========================================================

    loss = nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size),
        targets.reshape(-1)
    )


    # ========================================================
    # BACKPROP
    # ========================================================

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    # ========================================================
    # PRINT
    # ========================================================

    if step % 100 == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}")


print("Training complete.")

Saving input.txt to input.txt
Vocabulary size: 65
Dataset length: 1115394
Device: cuda
GPU: Tesla T4
Token embedding: cuda:0
Qc: cuda:0
Qc_2: cuda:0
FFN_1: cuda:0
FFN_2: cuda:0
LM head: cuda:0
Step 0: Loss = 4.5005
Step 100: Loss = 2.6194
Step 200: Loss = 2.4858
Step 300: Loss = 2.4552
Step 400: Loss = 2.3974
Step 500: Loss = 2.3314
Step 600: Loss = 2.2862
Step 700: Loss = 2.2257
Step 800: Loss = 2.1512
Step 900: Loss = 2.1261
Step 1000: Loss = 2.0852
Step 1100: Loss = 2.0758
Step 1200: Loss = 2.0200
Step 1300: Loss = 2.0114
Step 1400: Loss = 1.9622
Step 1500: Loss = 1.9865
Step 1600: Loss = 1.9261
Step 1700: Loss = 1.9941
Step 1800: Loss = 1.8852
Step 1900: Loss = 1.8804
Step 2000: Loss = 1.8918
Step 2100: Loss = 1.7693
Step 2200: Loss = 1.8823
Step 2300: Loss = 1.8196
Step 2400: Loss = 1.8098
Step 2500: Loss = 1.8269
Step 2600: Loss = 1.8090
Step 2700: Loss = 1.7767
Step 2800: Loss = 1.7607
Step 2900: Loss = 1.7812
Step 3000: Loss = 1.7602
Step 3100: Loss = 1.7191
Step 3200: Loss = 1

In [4]:
def generate_text(prompt, max_new_chars=300, temperature=1.0):

    x = torch.tensor([stoi[c] for c in prompt], dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_chars):

        x_cond = x[:, -train_length:]

        token_emb = token_embedding(x_cond)

        positions = torch.arange(x_cond.size(1), device=device)

        h = token_emb + pos_embedding(positions)

        Q_t = Qc(h)
        K_t = Kc(h)
        V_t = Vc(h)

        h_attn, _ = Attention_1(Q_t, K_t, V_t, return_scores=False)

        h = h + h_attn
        h = h + FFN_1(h)

        Q_t2 = Qc_2(h)
        K_t2 = Kc_2(h)
        V_t2 = Vc_2(h)

        h_attn2, _ = Attention_1(Q_t2, K_t2, V_t2, return_scores=False)

        h = h + h_attn2
        h = h + FFN_2(h)

        logits = lm_head(h[:, -1, :])
        logits = logits / temperature

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)

        x = torch.cat([x, next_id], dim=1)

    return "".join(itos[i.item()] for i in x[0])


print(generate_text("Hi sir! ", max_new_chars=20, temperature=0.1))

Hi sir! how I have stand the
